In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_excel('../../data/domain_sci_input/supplemenary_data.xlsx', sheet_name='510_selected_contigs')
df

,contig_id,sequence,corresponding_srr,assembler,cluster_membership,known_or_potentially_novel_tobamovirus,contig_length,orf1_complete,orf1_partial,orf1_length,...,first_blastx_hit_protein,study_accession,study_title,organism_name,submitter,country,collection_date,source_sample_category,source_sample_subcategory,genbank_accession_number
0,k119_47794_flag_1_multi_143.0000_len_5109_SRR5...,GGACAGCAAAAAGTGACAGGGACATTCGGTGTGCAAGATTTCTATA...,SRR5087336,megahit,NaN,False,5109,NaN,NaN,NaN,...,RNA-dependent RNA polymerase [Astegopteryx for...,SRP094753,IL10 and CpG stimulations of P493-6 cells,Homo sapiens,"Institute of Functional Genomics, University o...",Germany,NaN,Host-associated,Other,NaN
1,k141_112743_flag_1_multi_2.0000_len_612_SRR728...,GTACAGGCATATGATTAAAGCGCAACCGAAGCAGAAACTGGATCTG...,SRR7288019,megahit,NaN,True,612,NaN,NaN,NaN,...,"184 kDa replicase, partial [Tomato mosaic virus]",SRP150191,Viral metagenome in river water samples,freshwater metagenome,Jiangsu University,China:Jiangsu,2015,Aquatic,Freshwater,NaN
2,k141_1210_flag_1_multi_2.0000_len_787_SRR1784304,GGTCCGCGAGGTCGTGCTAGAGGTAAGTCTGGTGTTGATCGTAAGG...,SRR1784304,megahit,Cluster_2,True,787,0.0,0.0,0.0,...,18 kDa coat protein [Cucumber mottle virus],SRP037995,Amazon Continuum Metatranscriptomes,aquatic metagenome,University of Georgia,Brazil: Amazon River,2011-05-08,Aquatic,Freshwater,NaN
3,k141_12174_flag_1_multi_7.0000_len_1106_SRR178...,TTTTTCGGGCCGCTACCCGCGGTTCGGGGGGGATTCGAACCCCTAG...,SRR1784304,megahit,Cluster_4,True,1106,0.0,0.0,0.0,...,coat protein [Tropical soda apple mosaic virus],SRP037995,Amazon Continuum Metatranscriptomes,aquatic metagenome,University of Georgia,Brazil: Amazon River,2011-05-08,Aquatic,Freshwater,NaN
4,k141_135785_flag_1_multi_5.0000_len_1201_SRR72...,GATTTGATACAGTGGCGGACAACGAAGCTGCCAAACTGTCCACATA...,SRR7288019,megahit,NaN,True,1201,NaN,NaN,NaN,...,126 kDa protein [Tobacco mild green mosaic virus],SRP150191,Viral metagenome in river water samples,freshwater metagenome,Jiangsu University,China:Jiangsu,2015,Aquatic,Freshwater,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
505,NODE_9562_length_2589_cov_93.538279_SRR769314,GACATCGACAGGACAGCGGTACGAGATAAAAAGTGAACCGACAATT...,SRR769314,spades,NaN,False,2589,NaN,NaN,NaN,...,"MAG: RNA-dependent RNA polymerase, partial [Ut...",SRP019027,Transcriptome sequencing of neonatal thymic ep...,Mus musculus,Institute for Research in Immunology and Cance...,Canada,NaN,Host-associated,Animal,NaN
506,NODE_9608_length_2629_cov_6.756270_SRR10143187,CCACCGCTGTTTCAAAGGCACGCTTCGCATCACCTAAAACACTCAA...,SRR10143187,spades,NaN,False,2629,NaN,NaN,NaN,...,MAG: RNA-dependent RNA polymerase [Wufeng shre...,SRP222475,Ovis aries breed:bashibai Raw sequence reads,Ovis aries,Xinjiang Agricultural University,China,NaN,Host-associated,Animal,NaN
507,NODE_9819_length_675_cov_2.186131_SRR12904122,ATATAGTACAAAGGTATCTAACCTTGAAAACTTACAAAACCAAGTA...,SRR12904122,spades,NaN,True,675,NaN,NaN,NaN,...,"replication-associated protein, partial [Peppe...",SRP288687,Freshwater Metagenome,freshwater metagenome,Jiangsu University,China: Yangtze River,2017,Aquatic,Freshwater,NaN
508,NODE_9835_length_2717_cov_4.322727_SRR10255703,CTTATGGTGCTGAGGATTACCTTGAAAAATCCGATGATGAGCTCCT...,SRR10255703,spades,NaN,False,2717,NaN,NaN,NaN,...,2a protein [Cucumber mosaic virus],SRP225013,RNA-sequenceing for lfy1 near-isogenic lines,Zea mays,Chinese Academy of Agricultural Sciences,China,NaN,Host-associated,Plant,NaN


In [4]:
df['gt'] = np.where(df['ground_truth_category'].isin(['tob1', 'tob2', 'tob3']), 1, 0)

In [31]:
# Table: ORF presence vs model correctness
orf_map = {
    'ORF1': ('orf1_complete', 'orf1_partial'),
    'ORF2': ('orf2_rdrp_complete', 'orf2_rdrp_partial'),
    'ORF3': ('orf3_mp_complete', 'orf3_mp_partial'),
    'ORF4': ('orf4_cp_complete', 'orf4_cp_partial'),
}

false_breakdown_order = ['tob1', 'tob2', 'tob3', 'oth1', 'oth2', 'mas']

model_pred = pd.to_numeric(df['model_prediction'], errors='coerce')
gt = pd.to_numeric(df['gt'], errors='coerce')

model_values = set(model_pred.dropna().unique())
gt_values = set(gt.dropna().unique())

model_pred_for_compare = model_pred
is_correct = model_pred_for_compare.eq(gt)

ground_truth_category = (
    df['ground_truth_category']
    .fillna('')
    .astype(str)
    .str.strip()
    .str.lower()
)

rows = []
for orf_name, (complete_col, partial_col) in orf_map.items():
    has_complete = pd.to_numeric(df[complete_col], errors='coerce').fillna(0).gt(0)
    has_partial = pd.to_numeric(df[partial_col], errors='coerce').fillna(0).gt(0)
    has_orf = has_complete | has_partial

    contig_count = int(has_orf.sum())
    correct_count = int((has_orf & is_correct).sum())

    false_mask = has_orf & ~is_correct
    false_count = int(false_mask.sum())
    false_rate = false_count / contig_count if contig_count else np.nan

    false_breakdown = [
        int((false_mask & ground_truth_category.eq(category)).sum())
        for category in false_breakdown_order
    ]
    false_breakdown_str = f"{false_count}=[{'+'.join(map(str, false_breakdown))}]"

    rows.append(
        {
            'orf': orf_name,
            'orf_complete': int(has_complete.sum()),
            'orf_partial': int(has_partial.sum()),
            'orf_complete_or_partial': contig_count,
            'correct_predictions': correct_count,
            'false_predictions': false_count,
            'false_predictions_breakdown[tob1,tob2,tob3,oth1,oth2,mas]': false_breakdown_str,
            'false_prediction_rate': false_rate,
        }
    )

orf_summary = pd.DataFrame(rows)
orf_summary['false_prediction_rate_pct'] = (orf_summary['false_prediction_rate'] * 100).round(2)

# orf_summary.to_csv("error_rate.csv", index=False)
orf_summary

,orf,orf_complete,orf_partial,orf_complete_or_partial,correct_predictions,false_predictions,"false_predictions_breakdown[tob1,tob2,tob3,oth1,oth2,mas]",false_prediction_rate,false_prediction_rate_pct
0,ORF1,3,56,59,50,9,9=[0+1+8+0+0+0],0.152542,15.25
1,ORF2,7,46,53,53,0,0=[0+0+0+0+0+0],0.000000,0.00
2,ORF3,26,31,57,53,4,4=[0+3+1+0+0+0],0.070175,7.02
3,ORF4,51,18,69,68,1,1=[0+0+1+0+0+0],0.014493,1.45
